# Step 1a — Data Preparation

Load, filter, tokenize, and save the Reddit Mental Health dataset to Google Drive.
Run this notebook once; the training notebook (`step1b_train.ipynb`) loads from Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where you keep this project in your Drive
PROJECT_DIR = '/content/drive/MyDrive/CSC2412/dp-finetuned-llms'

import os
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets

## 3. Imports & Config

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import GPT2TokenizerFast

# ── Config ─────────────────────────────────────────────────────────────────────
SEED          = 42
MAX_LENGTH    = 128
TRAIN_SAMPLES = 10_000
VAL_SAMPLES   = 1_000

np.random.seed(SEED)

## 4. Load Dataset

In [ ]:
raw = load_dataset('solomonk/reddit_mental_health_posts', split='train')
print(f'Total examples: {len(raw)}')
print('Columns:', raw.column_names)
print('\nSample row:')
print(raw[0])

In [ ]:
# Identify the text column — adjust if needed after inspecting raw[0]
TEXT_COL = 'text'  # change to 'selftext', 'body', etc. if needed

# Filter out empty/short posts and shuffle
raw = raw.filter(lambda x: x[TEXT_COL] is not None and len(x[TEXT_COL].strip()) > 50)
raw = raw.shuffle(seed=SEED)

train_raw = raw.select(range(TRAIN_SAMPLES))
val_raw   = raw.select(range(TRAIN_SAMPLES, TRAIN_SAMPLES + VAL_SAMPLES))
print(f'Train: {len(train_raw)} | Val: {len(val_raw)}')

## 5. Tokenize

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

train_ds = train_raw.map(tokenize, batched=True, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(tokenize,   batched=True, remove_columns=val_raw.column_names)

print('Tokenization done.')
print('Train sample keys:', list(train_ds[0].keys()))

## 6. Save to Drive

In [ ]:
train_ds.save_to_disk(f'{PROJECT_DIR}/data/train_ds')
val_ds.save_to_disk(f'{PROJECT_DIR}/data/val_ds')
tokenizer.save_pretrained(f'{PROJECT_DIR}/data/tokenizer')

print('Saved:')
print(f'  {PROJECT_DIR}/data/train_ds')
print(f'  {PROJECT_DIR}/data/val_ds')
print(f'  {PROJECT_DIR}/data/tokenizer')

---
**Next:** Run `step1b_train.ipynb` to fine-tune GPT-2 on this data.